#1. Importing Packages

In [ ]:
!pip install sentence-transformers -q

In [ ]:
!pip install vaderSentiment -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 4.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from collections import Counter
import re
from sentence_transformers import SentenceTransformer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.decomposition import PCA
from math import radians, sin, cos, sqrt, atan2


----

#2. Loading clean data

In [ ]:
listings=pd.read_parquet('/content/data/listings_clean.parquet')
listings.head()

,id,name,description,picture_url,host_id,host_name,host_since,host_response_time,host_response_rate,host_acceptance_rate,...,reviews_per_month,log_price,bathrooms_parsed,has_reviews,has_neighborhood_overview,host_location_given,has_host_about,has_description,host_effort_score,days_since_last_review
0,9991,Geourgeous flat - outstanding views,4 bedroom with very large windows and outstand...,https://a0.muscache.com/pictures/42799131/59c8...,33852,Philipp,2009-08-25,a few days or more,20.0,0.0,...,0.06,4.912655,2.5,1,1,1,1,1,5,2351.0
1,20858,Designer Loft in Berlin Mitte,Bright and sunny condo with two balconies in a...,https://a0.muscache.com/pictures/108232/205b19...,71331,Marc,2010-01-18,within an hour,100.0,96.0,...,0.89,5.313206,1.0,1,1,1,1,1,5,331.0
2,22677,"""Prenzel Garten"" mit Terrasse",Relax in our little garden after exploring Ber...,https://a0.muscache.com/pictures/miso/Hosting-...,87357,Ramfis,2010-03-02,within a few hours,100.0,98.0,...,3.29,5.030438,1.0,1,1,1,1,1,5,266.0
3,30295,PEACEFUL FLAT WITH VINTAGE TOUCH IN PRENZLAUER...,"Right in the heart of Prenzlauer Berg, possibl...",https://a0.muscache.com/pictures/4551295/95263...,485838,Michele,2011-04-05,within an hour,100.0,99.0,...,2.75,5.153292,1.0,1,1,1,1,1,5,267.0
4,37004,WONDERFUL ROOM for a bubble in Berlin Central,Great apartment in the style of a typical late...,https://a0.muscache.com/pictures/209262/c53bb3...,159734,Bernd,2010-07-06,within a day,75.0,57.0,...,0.21,4.110874,1.0,1,1,1,1,1,5,304.0


----

#3. Hot encoding Categorical data

##3.1. Listings

### Encode categoricals

In [ ]:
# room_type
listings = pd.get_dummies(listings, columns=['room_type'], drop_first=True)

# neighbourhood_cleansed
listings = pd.get_dummies(listings, columns=['neighbourhood_cleansed'], drop_first=True)

# host_response_time
listings = pd.get_dummies(listings, columns=['host_response_time'], drop_first=True)

### Parse amenities

In [ ]:
## Parsing list of strings ##
def parse_amenities(text):
    try:
        return ast.literal_eval(text)
    except:
        return []

listings['amenities_parsed'] = listings['amenities'].apply(parse_amenities)

## most common amenities ##
all_amenities = [a for sublist in listings['amenities_parsed'] for a in sublist]
total = len(listings)
amenity_pcts = {a: count/total*100 for a, count in Counter(all_amenities).most_common(50)}

for amenity, pct in amenity_pcts.items():
    print(f"{pct:5.1f}%  {amenity}")


 89.3%  Wifi
 87.8%  Smoke alarm
 86.6%  Kitchen
 84.6%  Hot water
 81.5%  Hair dryer
 79.4%  Dishes and silverware
 77.5%  Hangers
 77.1%  Bed linens
 74.4%  Essentials
 73.2%  Cooking basics
 70.4%  Refrigerator
 64.3%  Iron
 62.8%  Dedicated workspace
 61.6%  Hot water kettle
 58.8%  Heating
 52.9%  Dining table
 50.0%  Room-darkening shades
 48.7%  Wine glasses
 48.3%  Shampoo
 48.1%  Dishwasher
 46.8%  Self check-in
 45.5%  Toaster
 45.0%  Long term stays allowed
 45.0%  Shower gel
 44.5%  TV
 44.4%  Washer
 44.0%  Fire extinguisher
 43.1%  Oven
 42.6%  Body soap
 42.1%  Cleaning products
 41.7%  Freezer
 40.5%  Coffee maker
 39.5%  Microwave
 39.0%  Stove
 37.5%  First aid kit
 37.0%  Extra pillows and blankets
 35.8%  Coffee
 34.7%  Drying rack for clothing
 34.0%  Baking sheet
 32.0%  Luggage dropoff allowed
 30.3%  Bathtub
 29.9%  Central heating
 29.1%  Carbon monoxide alarm
 28.5%  Private entrance
 26.6%  Free street parking
 25.6%  Lockbox
 25.3%  Elevator
 24.1%  Clothing

In [ ]:
## Selecting Useful amenities ##
#anything above 70% (Wifi, Smoke alarm, Kitchen, Hot water, Hair dryer) no discriminating power.
selected_amenities = [
    'Dishwasher',          # comfort/quality signal
    'Self check-in',       # convenience
    'TV',                  # comfort
    'Washer',              # comfort
    'Private entrance',    # quality
    'Elevator',            # comfort
    'Dedicated workspace', # comfort
    'Long term stays allowed', # target audience
    'Free street parking', # convenience
    'Shampoo',             # comfort
    'Toaster',             # quality
    'Iron',                # comfort
]

for amenity in selected_amenities:
    col_name = 'amenity_' + amenity.lower().replace(' ', '_').replace('-', '_')
    listings[col_name] = listings['amenities_parsed'].apply(lambda x: int(amenity in x))


### Count amenities as a feature

In [ ]:
listings['amenities_count'] = listings['amenities_parsed'].apply(len)

# check corr with price
print(listings[['amenities_count', 'log_price']].corr())

                 amenities_count  log_price
amenities_count         1.000000   0.226841
log_price               0.226841   1.000000


### Extract host_since as experience

In [ ]:
listings['host_since'] = pd.to_datetime(listings['host_since'])
listings['host_tenure_days'] = (pd.Timestamp.today() - listings['host_since']).dt.days
listings.drop(columns=['host_since'], inplace=True)

### Dropping unprocessed columns

In [ ]:
cols_to_drop = [
    'amenities',
    'amenities_parsed',
    'name',
    'picture_url',
    'host_verifications',
]
listings.drop(columns=cols_to_drop, inplace=True)
print(listings.shape)
listings.head()

(6981, 208)


,id,description,host_id,host_name,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_total_listings_count,host_has_profile_pic,...,amenity_private_entrance,amenity_elevator,amenity_dedicated_workspace,amenity_long_term_stays_allowed,amenity_free_street_parking,amenity_shampoo,amenity_toaster,amenity_iron,amenities_count,host_tenure_days
0,9991,4 bedroom with very large windows and outstand...,33852,Philipp,20.0,0.0,0.0,1.0,1.0,1.0,...,0,1,0,0,0,0,0,1,32,6135
1,20858,Bright and sunny condo with two balconies in a...,71331,Marc,100.0,96.0,0.0,3.0,3.0,1.0,...,0,1,0,1,0,0,0,1,30,5989
2,22677,Relax in our little garden after exploring Ber...,87357,Ramfis,100.0,98.0,1.0,1.0,1.0,1.0,...,0,0,1,0,0,0,1,1,40,5946
3,30295,"Right in the heart of Prenzlauer Berg, possibl...",485838,Michele,100.0,99.0,1.0,4.0,5.0,1.0,...,1,1,1,0,0,1,0,1,31,5547
4,37004,Great apartment in the style of a typical late...,159734,Bernd,75.0,57.0,0.0,1.0,1.0,1.0,...,1,0,1,1,1,0,0,1,35,5820


----

#4. Processing Text

##4.1. Listings

In [ ]:
# description length
listings['description_length'] = listings['description'].apply(len)


### Checking dominant Language

In [ ]:
!pip install langdetect -q
from langdetect import detect

def detect_lang(text):
    try:
        return detect(text)
    except:
        return 'unknown'

sample = listings['description'].dropna()
langs = sample.apply(detect_lang)
print(langs.value_counts())

description
en         6699
unknown     232
de           28
nl           10
it            4
fr            2
da            1
af            1
ru            1
tr            1
id            1
ro            1
Name: count, dtype: int64


### Sentiment

In [ ]:
analyzer = SentimentIntensityAnalyzer()

listings['description_sentiment'] = listings['description'].apply(
    lambda x: analyzer.polarity_scores(x)['compound'] if x else 0
)
print(listings['description_sentiment'].describe())

description_sentiment    0.087673
log_price                1.000000
Name: log_price, dtype: float64
count    6981.000000
mean        0.710964
std         0.323741
min        -0.941300
25%         0.585900
50%         0.858500
75%         0.939200
max         0.994000
Name: description_sentiment, dtype: float64


### Embeddings

In [ ]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

embeddings = model.encode(
    listings['description'].tolist(),
    batch_size=64,
    show_progress_bar=True
)

print(embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/110 [00:00<?, ?it/s]

(6981, 384)


In [ ]:
## Reduction ##
pca = PCA(n_components=50, random_state=42)
embeddings_reduced = pca.fit_transform(embeddings)

print(f"Variance explained: {pca.explained_variance_ratio_.sum():.2%}")

emb_cols = [f'desc_emb_{i}' for i in range(50)]
df_embeddings = pd.DataFrame(embeddings_reduced, columns=emb_cols, index=listings.index)
listings = pd.concat([listings, df_embeddings], axis=1)

Variance explained: 77.90%


In [ ]:
## Saving ##
np.save('description_embeddings.npy', embeddings)
np.save('description_embeddings_pca50.npy', embeddings_reduced)
listings.to_parquet('listings_features.parquet', index=False)

In [ ]:
listings.shape

(6981, 260)

-----

#5. Processing Spatial Data

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # earth radius in km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

In [ ]:
# Berlin landmarks (lat, lon)
landmarks = {
    'dist_city_center':  (52.5200, 13.4050),  # Alexanderplatz
    'dist_ber_airport':  (52.3667, 13.5033),  # BER airport
    'dist_kreuzberg':    (52.4988, 13.4028),  # Kreuzberg
}

In [ ]:
for col_name, (lat, lon) in landmarks.items():
    listings[col_name] = listings.apply(
        lambda row: haversine(row['latitude'], row['longitude'], lat, lon),
        axis=1
    )

In [ ]:
# correlations
dist_cols = list(landmarks.keys())
print(listings[dist_cols + ['log_price']].corr()['log_price'].drop('log_price').round(3))

dist_city_center   -0.147
dist_ber_airport    0.048
dist_kreuzberg     -0.101
Name: log_price, dtype: float64


In [ ]:
## Saving ##
listings.drop(columns=['dist_ber_airport'], inplace=True)

listings.to_parquet('listings_features.parquet', index=False)
listings.shape

(6981, 262)

-----

#6. Processing Reviews

In [ ]:
!wget 'https://data.insideairbnb.com/germany/be/berlin/2025-09-23/data/reviews.csv.gz'

--2026-06-12 18:35:58--  https://data.insideairbnb.com/germany/be/berlin/2025-09-23/data/reviews.csv.gz
Resolving data.insideairbnb.com (data.insideairbnb.com)... 143.204.204.129, 143.204.204.13, 143.204.204.47, ...
Connecting to data.insideairbnb.com (data.insideairbnb.com)|143.204.204.129|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 72609019 (69M) [application/x-gzip]
Saving to: ‘reviews.csv.gz’

reviews.csv.gz      100%[===================>]  69.25M  29.3MB/s    in 2.4s    

2026-06-12 18:36:01 (29.3 MB/s) - ‘reviews.csv.gz’ saved [72609019/72609019]



In [ ]:
## Filtering only kept rows
reviews = pd.read_csv('reviews.csv.gz', compression='gzip')

valid_ids = listings['id'].unique()
reviews_filtered = reviews[reviews['listing_id'].isin(valid_ids)]

print(f"All reviews: {len(reviews)}")
print(f"Filtered reviews: {len(reviews_filtered)}")

All reviews: 635471
Filtered reviews: 533090


In [ ]:
reviews_filtered['sentiment'] = reviews_filtered['comments'].apply(
    lambda x: analyzer.polarity_scores(str(x))['compound']
)

review_features = reviews_filtered.groupby('listing_id').agg(
    review_sentiment_mean=('sentiment', 'mean'),
    review_sentiment_std=('sentiment', 'std'),
).reset_index()

# correlation
df_temp = listings.merge(review_features, left_on='id', right_on='listing_id', how='left')

print(df_temp[['review_sentiment_mean', 'review_sentiment_std', 'log_price']].corr()['log_price'])

review_sentiment_mean    0.083726
review_sentiment_std    -0.131266
log_price                1.000000
Name: log_price, dtype: float64


/tmp/ipykernel_685/2399700920.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reviews_filtered['sentiment'] = reviews_filtered['comments'].apply(


In [ ]:
listings = listings.merge(review_features, left_on='id', right_on='listing_id', how='left')

# nulls for listings with no reviews
listings['review_sentiment_mean'] = listings['review_sentiment_mean'].fillna(0)
listings['review_sentiment_std'] = listings['review_sentiment_std'].fillna(0)

# duplicate listing_id column from merge
listings.drop(columns=['listing_id'], inplace=True)

print(listings.shape)

(6981, 264)


In [ ]:
listings.to_parquet('listings_features.parquet', index=False)
